# 02 — Clause Segmentation
This notebook explores how ClauseGuard splits a full contract into individual clauses.

**Two strategies are used:**
1. **Regex-based** — detect numbered/markdown section headers (e.g. `## 1. Services`)
2. **spaCy sentence boundary detection** — fallback if no numbered structure is found

**Production file:** `backend/segmenter.py`

In [ ]:
import re
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded:", nlp.meta['name'])

## 1. Strategy A — Numbered / Markdown Header Detection

In [ ]:
def detect_numbered_clauses(text):
    """
    Detects clause boundaries using patterns like:
      - ## 1. Services
      - Section 1. Payment
      - Article 2. Termination
    """
    pattern = r'(?:\n|^)\s*(?:##\s*|Section\s+|Article\s+)?(\d+\.\s+)'
    matches = list(re.finditer(pattern, text))
    if not matches:
        return []

    clauses = []
    preamble = text[:matches[0].start()].strip()
    if preamble:
        clauses.append(preamble)

    for i, match in enumerate(matches):
        start = match.start()
        if match.group(0).startswith('\n'):
            start += 1
        end = matches[i+1].start() if i + 1 < len(matches) else len(text)
        clause_content = text[start:end].strip()
        if clause_content:
            clauses.append(clause_content)

    return clauses

sample_text = """This is a sample contract.

## 1. Services
The vendor agrees to provide services.

## 2. Payment
Payment shall be made within 30 days.
"""
clauses = detect_numbered_clauses(sample_text)
print(f"Found {len(clauses)} numbered clauses\n")
for i, c in enumerate(clauses):
    print(f"--- Clause {i} ---\n{c}\n")

## 2. Strategy B — spaCy Sentence Boundary Detection (fallback)

In [ ]:
def split_into_sentences(text):
    """Use spaCy's statistical model to split text into sentences."""
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

unnumbered_text = """
The vendor agrees to provide services as described. Payment shall be made within 30 days.
All intellectual property shall transfer to the client upon full payment.
Either party may terminate this agreement with 30 days notice.
"""

sentences = split_into_sentences(unnumbered_text)
print(f"spaCy found {len(sentences)} sentences:")
for i, s in enumerate(sentences, 1):
    print(f"  {i}. {s}")